# NYC Taxi Fare & Duration
## Exploratory Data Analysis (EDA)

In [1]:
%load_ext autoreload
%autoreload 2

Imports

In [2]:
import os
import sys
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import wget

sys.path.append("..")
import source.configs as configs

# Check if we are in COLAB
IN_COLAB = 'google.colab' in sys.modules

In [3]:
if not os.path.exists("../dataset/yellow_tripdata_2022-05.parquet"):
    if IN_COLAB:
        !wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2022-05.parquet "../dataset"
    else:
        wget.download(configs.START_DATASET_URL, "../dataset")

AttributeError: module 'source.configs' has no attribute 'START_DATASET_URL'

## Read Dataset

In [ ]:
dataset = pd.read_parquet("../dataset/yellow_tripdata_2022-05.parquet")

### List of columns

In [ ]:
dataset.columns.to_list()

### Types of columns

In [ ]:
dataset.dtypes

Conclusions
* We have tpep_pickup_datetime, tpep_dropoff_datetime to calculate trip duration.


### Head / Describe

In [ ]:
dataset.head()

### Relevant features
* trip_distance is a very important feature to predict duration of the trip.
* Trip duration is no in the dataset, but is a target feature we must build using pick-up and drop-off timestamps.
* fare_amount is one of the target variables.
* passenger_count could be relevant.
* PULocationID, DOLocationID, these describe taxi zones, they can relevant.
* RateCodeID: Tells which rate was applied at the end of the trip. It can be relevant.
* The fare amout only depends on duration/distance of the trip. There are other charges that go into total amount.

Check this web page:
https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf

Total amount = Fare + Overnight_Charges + RushHour_Charges + ...

In [ ]:
dataset.describe()

In [ ]:
for col in ["VendorID", "RatecodeID", "PULocationID", "DOLocationID",  "payment_type", "passenger_count", "extra", "mta_tax"]:
    print(f"{col}: {dataset[col].nunique()}")

### Trip duration feature

https://pandas.pydata.org/docs/reference/api/pandas.Timedelta.html

https://pandas.pydata.org/docs/user_guide/timeseries.html

In [ ]:
dataset["trip_duration"] = (dataset["tpep_dropoff_datetime"]-dataset["tpep_pickup_datetime"]).dt.total_seconds()/60

### Percentiles

In [ ]:
for p in [0.9, 0.95, 0.99, 0.995]:
    print(f"** Percentiles for {p} **")
    for col in ["trip_distance", "fare_amount", "total_amount", "trip_duration"]:
        print(f"  {col} {dataset[col].quantile(p)}")

### Histogram of trip duration

In [ ]:
plt.figure(figsize=(12,5))
plt.hist(dataset["trip_duration"], bins=20, range=(0,100))

### Histogram of trip distance

In [ ]:
plt.figure(figsize=(12,5))
plt.hist(dataset["trip_distance"], bins=25, range=(0,25))

### Histogram of day of week

In [ ]:
#for col in ["", "tpep_dropoff_datetime", "passenger_count", "",  "payment_type", "PULocationID"]:
plt.hist(dataset["tpep_pickup_datetime"].dt.day_of_week,bins=7)

### Histogram of hour of day

In [ ]:
plt.hist(dataset["tpep_pickup_datetime"].dt.hour, bins=24)

### Histogram of fare amount

In [ ]:
plt.figure(figsize=(10,5))
plt.hist(dataset["fare_amount"], bins=20, range=(0,80))

### Histogram of passenger count

In [ ]:
plt.figure(figsize=(12,5))
plt.hist(dataset["passenger_count"], bins=10, range=(0,8))

### Histogram of RatecodeID

In [ ]:
plt.hist(dataset["RatecodeID"], bins=8, range=(0,6))

### Analyze NA values

In [ ]:
na_count = dataset.isna().sum()
total_entries = dataset.shape[0]
na_percentage = (na_count / total_entries) * 100
na_summary = pd.DataFrame({'NA Count': na_count, 'NA Percentage': na_percentage})
print(na_summary)